In [ ]:
from transformers import pipeline 
from tqdm import tqdm
import os 
import pandas as pd 


os.environ["TRANSFORMERS_OFFLINE"] = "1"

###  name for your file here 
filename = "fileName"

MODEL_DIR = r"C:\Users\rebecca.gray\Documents\Requests\jake\editorial multilingual sentiment\sentiment_model"

#### file path here 
EXCEL_PATH = r"file\path"
ARTICLE_COL = "Snippet"
OUTPUT_PATH = "scored sentiment " + filename + ".xlsx"
BATCH_SIZE = 32

In [ ]:
# load local model
pipe = pipeline(
    "text-classification",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    truncation=True,
    max_length=800
)

# load Excel
df = pd.read_excel(EXCEL_PATH, sheet_name="Articles")
print(f"Loaded {len(df)} rows")

In [ ]:
# clean data
## by splitting out english language/pre-rated rows
SENTIMENT_COL = "Sentiment"  

prerated = df[df[SENTIMENT_COL].notna()].copy()
df = df[df[SENTIMENT_COL].isna()].reset_index(drop=True)

print(f"Pre-rated rows (skipping): {len(prerated)}")
print(f"Rows to score: {len(df)}") 

In [ ]:
# score in batches 
## some files will need lots of processing, this allows
## for stages of progress 

texts = df[ARTICLE_COL].tolist()
labels, scores = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = [str(t) if not pd.isna(t) and str(t).strip() != "" else "" for t in texts[i:i+BATCH_SIZE]]
    results = pipe(batch, truncation=True, max_length=800)
    for j, r in enumerate(results):
        if batch[j] == "":
            labels.append(None)
            scores.append(None)
        else:
            labels.append(r["label"])
            scores.append(round(r["score"], 4))


In [ ]:
# save final output 

df["Sentiment"] = labels
df["Sentiment_Score"] = scores

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Articles", index=False)
    prerated.to_excel(writer, sheet_name="prerated", index=False)

print(f"Done! Results saved to {OUTPUT_PATH}")
print(f"  - Articles sheet: {len(df)} scored rows")
print(f"  - prerated sheet: {len(prerated)} skipped rows")